
# DATA 304 — Session 2 Demo: Representing Text Numerically

This notebook demonstrates:
- Bag-of-Words (BoW)
- TF-IDF
- N-grams
- Exploring vocabulary and IDF
- **Keyword extraction per document (top-k)**  ← include in demo
- Cosine similarity
- PCA visualization of TF-IDF
- Saving sparse matrices


In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from scipy import sparse

import matplotlib.pyplot as plt

docs = [
    "Data wrangling with Python for messy data",
    "Python handles messy data and supports data analysis",
    "Wrangling text data requires cleaning and tokenization",
    "Machine learning uses numeric features like TF IDF",
    "Clustering documents with cosine similarity requires vectors"
]

print(f"#documents: {len(docs)}")

## Bag-of-Words (BoW)

In [ ]:
bow = CountVectorizer()
X_bow = bow.fit_transform(docs)

bow_df = pd.DataFrame(X_bow.toarray(), columns=bow.get_feature_names_out())
bow_df

## TF-IDF (with English stopwords)

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')
X_tfidf = tfidf.fit_transform(docs)

tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out()).round(3)
tfidf_df

## Stopwords

In [ ]:
# 1) Built-in English
tfidf_default = TfidfVectorizer(stop_words='english')
X_default = tfidf_default.fit_transform(docs)

# 2) Simple custom list
simple_stops = ['data', 'python', 'and']
tfidf_simple = TfidfVectorizer(stop_words=simple_stops)
X_simple = tfidf_simple.fit_transform(docs)

# 3) Customized English: add + remove, then convert to LIST
extra_words = {'data', 'python'}        # add
kept_words  = {'against', 'under'}      # keep (remove from stop list)
custom_stops_set = (ENGLISH_STOP_WORDS.union(extra_words)).difference(kept_words)
custom_stops = list(custom_stops_set)

tfidf_custom = TfidfVectorizer(stop_words=custom_stops)
X_custom = tfidf_custom.fit_transform(docs)

print("Default English:", len(tfidf_default.get_feature_names_out()), "terms")
print("Simple custom:", len(tfidf_simple.get_feature_names_out()), "terms")
print("Customized English:", len(tfidf_custom.get_feature_names_out()), "terms")
print("Added words:", extra_words)
print("Kept words:", kept_words)

## N-grams: unigrams + bigrams

In [ ]:
tfidf_uni_bi = TfidfVectorizer(stop_words='english', ngram_range=(1,2))
X_uni_bi = tfidf_uni_bi.fit_transform(docs)

# Show a sample of features to keep output readable
features = tfidf_uni_bi.get_feature_names_out()
pd.DataFrame({'feature': features}).head(20)

## Exploring vocabulary importance (IDF)

In [ ]:
idf = tfidf.idf_
vocab = tfidf.get_feature_names_out()
idf_df = pd.DataFrame({'term': vocab, 'idf': idf}).sort_values('idf', ascending=False)
idf_df.head(20)

## Keyword extraction per document (top-k by TF-IDF)

In [ ]:
k = 5
feature_array = np.array(tfidf.get_feature_names_out())
tfidf_matrix = X_tfidf.toarray()

for i, row in enumerate(tfidf_matrix, start=1):
    top_idx = row.argsort()[-k:][::-1]
    top_terms = feature_array[top_idx]
    top_scores = row[top_idx]
    print(f"Doc {i}:", list(zip(top_terms, np.round(top_scores, 3))))

## Cosine similarity of documents (using TF-IDF)

In [ ]:
sim = cosine_similarity(X_tfidf)
sim_df = pd.DataFrame(sim, index=[f"Doc{i}" for i in range(1, len(docs)+1)],
                           columns=[f"Doc{i}" for i in range(1, len(docs)+1)]).round(3)
sim_df

## PCA visualization of TF-IDF

In [ ]:
coords = PCA(n_components=2).fit_transform(X_tfidf.toarray())
plt.figure()
plt.scatter(coords[:, 0], coords[:, 1])
for i, (x, y) in enumerate(coords, start=1):
    plt.text(x, y, f"Doc{i}")
plt.title("PCA projection of TF-IDF (2D)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

## Saving sparse TF-IDF to disk (.npz)

In [ ]:
# --- Temporary scaling to illustrate memory effect ---
# Duplicate the corpus so the TF-IDF matrix becomes larger
docs_large = docs * 500

# Recompute TF-IDF on the larger corpus
tfidf_large = TfidfVectorizer(stop_words="english", max_features=5000)
X_large = tfidf_large.fit_transform(docs_large)

# Show the matrix type
print("Matrix type:", type(X_large))
print("Matrix shape:", X_large.shape)

# --- Save dense vs. sparse representations ---
np.save("./data/tfidf_dense.npy", X_large.toarray())   # dense NumPy array
np.save("./data/tfidf_object.npy", X_large)            # pickled sparse object
sparse.save_npz("./data/tfidf_sparse.npz", X_large)    # optimized sparse format

# --- Compare approximate memory usage in RAM ---
dense_kb = (X_large.toarray().nbytes) / 1024
sparse_kb = (X_large.data.nbytes + X_large.indices.nbytes + X_large.indptr.nbytes) / 1024

print("\nRAM")
print(f"Dense RAM usage:  {dense_kb:.1f} KB")
print(f"Sparse RAM usage: {sparse_kb:.1f} KB")

# --- Compare file sizes on disk ---
dense_kb = os.path.getsize("./data/tfidf_dense.npy") / 1024
object_kb = os.path.getsize("./data/tfidf_object.npy") / 1024
sparse_kb = os.path.getsize("./data/tfidf_sparse.npz") / 1024

print("\nOn Disk")
print(f"Dense file:  {dense_kb:.1f} KB")
print(f"Object file: {object_kb:.1f} KB")
print(f"Sparse file: {sparse_kb:.1f} KB")